
# Phase 1: Data Cleaning and Quality Checks

## 1. Loading the Raw Datasets

In this section, I will load all eight raw CSV datasets using pandas. I will inspect the available files and verify that the datasets can be loaded successfully before performing data quality checks.


In [83]:

from pathlib import Path
import pandas as pd

raw_data_path = Path("../data/raw")

csv_files = sorted(raw_data_path.glob("*.csv"))

print("Number of CSV files found:", len(csv_files))
print("\nCSV files:")
for file in csv_files:
    print(file.name)


Number of CSV files found: 8

CSV files:
customers DA.csv
inventory DA.csv
order_items DA.csv
orders DA.csv
payments DA.csv
products DA.csv
returns DA.csv
suppliers DA.csv


In [84]:

customers = pd.read_csv(raw_data_path / "customers DA.csv")
inventory = pd.read_csv(raw_data_path / "inventory DA.csv")
order_items = pd.read_csv(raw_data_path / "order_items DA.csv")
orders = pd.read_csv(raw_data_path / "orders DA.csv")
payments = pd.read_csv(raw_data_path / "payments DA.csv")
products = pd.read_csv(raw_data_path / "products DA.csv")
returns = pd.read_csv(raw_data_path / "returns DA.csv")
suppliers = pd.read_csv(raw_data_path / "suppliers DA.csv")

print("All eight datasets loaded successfully!")


All eight datasets loaded successfully!


NOW LETS CHECK ROWS, COLUMNS, AND DATA TYPES

In [85]:

datasets = {
    "customers": customers,
    "inventory": inventory,
    "order_items": order_items,
    "orders": orders,
    "payments": payments,
    "products": products,
    "returns": returns,
    "suppliers": suppliers
}

for name, df in datasets.items():
    print(f"\nDataset: {name}")
    print(f"Rows: {df.shape[0]}")
    print(f"Columns: {df.shape[1]}")
    print(df.dtypes)



Dataset: customers
Rows: 10000
Columns: 17
customer_id              str
first_name               str
last_name                str
email                    str
phone                  int64
city                     str
state                    str
pincode                int64
gender                   str
age                    int64
segment                  str
registration_date        str
last_login_date          str
is_verified            int64
is_active              int64
total_orders           int64
total_spent          float64
dtype: object

Dataset: inventory
Rows: 1000
Columns: 11
inventory_id               str
product_id                 str
warehouse_location         str
quantity_available       int64
quantity_reserved        int64
reorder_level            int64
reorder_quantity         int64
last_restocked_date        str
unit_cost                int64
total_inventory_value    int64
status                     str
dtype: object

Dataset: order_items
Rows: 100000
Columns: 11
item

CHECK THE MISSING VALUES

In [86]:
for name, df in datasets.items():
    print(f"\nDataset: {name}")
    print(df.isnull().sum())


Dataset: customers
customer_id          0
first_name           0
last_name            0
email                0
phone                0
city                 0
state                0
pincode              0
gender               0
age                  0
segment              0
registration_date    0
last_login_date      0
is_verified          0
is_active            0
total_orders         0
total_spent          0
dtype: int64

Dataset: inventory
inventory_id             0
product_id               0
warehouse_location       0
quantity_available       0
quantity_reserved        0
reorder_level            0
reorder_quantity         0
last_restocked_date      0
unit_cost                0
total_inventory_value    0
status                   0
dtype: int64

Dataset: order_items
item_id            0
order_id           0
product_id         0
product_name       0
category           0
quantity           0
unit_price         0
gst_rate           0
gst_amount         0
discount_amount    0
total_price   

LET'S CHECK DUPLICATE ROWS

In [87]:

for name, df in datasets.items():
    print(f"{name}: {df.duplicated().sum()} duplicate rows")


customers: 0 duplicate rows
inventory: 0 duplicate rows
order_items: 0 duplicate rows
orders: 0 duplicate rows
payments: 0 duplicate rows
products: 0 duplicate rows
returns: 0 duplicate rows
suppliers: 0 duplicate rows


ZERO DUPLICATE ROWS DOES NOT GUARANTEE UNIQUE IDS. FOR EXAMPLE, TWO DIFFERENT ROWS COULD STILL CONTAIN THE SAME ORDER_ID WHILE DIFFERING IN OTHER COLUMNS. WE'LL NOW CHECK UNIQUE IDs

In [88]:

key_columns = {
    "customers": "customer_id",
    "inventory": "inventory_id",
    "order_items": "item_id",
    "orders": "order_id",
    "payments": "payment_id",
    "products": "product_id",
    "returns": "return_id",
    "suppliers": "supplier_id"
}

for name, column in key_columns.items():
    df = datasets[name]
    print(f"{name}: {df[column].nunique()} unique IDs out of {len(df)} rows")


customers: 10000 unique IDs out of 10000 rows
inventory: 1000 unique IDs out of 1000 rows
order_items: 100000 unique IDs out of 100000 rows
orders: 50000 unique IDs out of 50000 rows
payments: 50000 unique IDs out of 50000 rows
products: 1000 unique IDs out of 1000 rows
returns: 10000 unique IDs out of 10000 rows
suppliers: 200 unique IDs out of 200 rows


SEVERAL COLUMNS THAT REPRESENT DATES ARE CURRENTLY STORED AS STRINGS. BEFORE CONVERTING THEM, LET'S INSPECT A FEW SAMPLE VALUES TO UNDERSTAND THEIR FORMATS. SO LET'S CHECK DATE FORMATS

In [89]:

date_columns = {
    "customers": ["registration_date", "last_login_date"],
    "inventory": ["last_restocked_date"],
    "orders": ["order_date", "delivered_date"],
    "payments": ["payment_date", "refund_date"],
    "products": ["launch_date"],
    "returns": ["return_date", "refund_date"],
    "suppliers": ["created_date"]
}

for name, columns in date_columns.items():
    print(f"\nDataset: {name}")
    for column in columns:
        print(f"{column}: {datasets[name][column].head(5).tolist()}")



Dataset: customers
registration_date: ['30-07-2024', '05-07-2023', '14-05-2022', '21-01-2025', '06-06-2023']
last_login_date: ['10-04-2025', '03-05-2024', '31-10-2023', '22-05-2025', '29-03-2024']

Dataset: inventory
last_restocked_date: ['11-09-2023', '10-03-2025', '22-11-2024', '10-11-2024', '05-08-2024']

Dataset: orders
order_date: ['22-05-2025', '16-10-2023', '08-10-2023', '02-10-2023', '15-06-2024']
delivered_date: [nan, '19-10-2023', nan, '09-10-2023', '22-06-2024']

Dataset: payments
payment_date: ['22-05-2025', '16-10-2023', '08-10-2023', '02-10-2023', '15-06-2024']
refund_date: [nan, nan, nan, nan, nan]

Dataset: products
launch_date: ['29-01-2025', '17-05-2024', '28-12-2024', '04-05-2023', '27-11-2023']

Dataset: returns
return_date: ['23-12-2024', '01-11-2023', '01-05-2025', '20-06-2025', '16-03-2025']
refund_date: ['30-12-2024', '06-11-2023', '06-05-2025', '24-06-2025', '23-03-2025']

Dataset: suppliers
created_date: ['22-05-2023', '25-03-2023', '27-12-2022', '15-01-2023'

LET'S CONVERT DATE COLUMNS TO DATETIME

In [90]:

for name, columns in date_columns.items():
    for column in columns:
        datasets[name][column] = pd.to_datetime(
            datasets[name][column],
            format="%d-%m-%Y",
            errors="coerce"
        )

for name, columns in date_columns.items():
    print(f"\nDataset: {name}")
    for column in columns:
        print(f"{column}: {datasets[name][column].dtype}")



Dataset: customers
registration_date: datetime64[us]
last_login_date: datetime64[us]

Dataset: inventory
last_restocked_date: datetime64[us]

Dataset: orders
order_date: datetime64[us]
delivered_date: datetime64[us]

Dataset: payments
payment_date: datetime64[us]
refund_date: datetime64[s]

Dataset: products
launch_date: datetime64[us]

Dataset: returns
return_date: datetime64[us]
refund_date: datetime64[us]

Dataset: suppliers
created_date: datetime64[us]


LET'S VERIFY INVALID DATES AFTER CONVERSION

In [91]:

for name, columns in date_columns.items():
    for column in columns:
        missing_count = datasets[name][column].isna().sum()
        print(f"{name}.{column}: {missing_count} missing dates")


customers.registration_date: 0 missing dates
customers.last_login_date: 0 missing dates
inventory.last_restocked_date: 0 missing dates
orders.order_date: 0 missing dates
orders.delivered_date: 17501 missing dates
payments.payment_date: 0 missing dates
payments.refund_date: 50000 missing dates
products.launch_date: 0 missing dates
returns.return_date: 0 missing dates
returns.refund_date: 0 missing dates
suppliers.created_date: 0 missing dates


NEXT, WE'LL EXAMINE ORDER STATUSES TO UNDERSTAND WHY SOME ORDERS HAVE NO DELIVERY DATE.

In [92]:

print(orders["status"].value_counts(dropna=False))


status
Delivered     32499
Cancelled      5996
Shipped        5071
Returned       3933
Processing     2501
Name: count, dtype: int64


WHAT THIS TELLS US:

- DELIVERED: THESE ORDERS WOULD GENERALLY BE EXPECTED TO HAVE A DELIVERY DATE.
- CANCELLED, SHIPPED, PROCESSING: MISSING DELIVERY DATES MAY BE LEGITIMATE, DEPENDING ON THE ORDER'S PROGRESS.
- RETURNED: THESE ORDERS MAY HAVE BEEN DELIVERED BEFORE BEING RETURNED, SO THEIR DELIVERY DATES DESERVE ATTENTION.

LET'S CHECK DELIVERY DATES AGAINST ORDER STATUS


In [93]:

print("Missing delivery dates by order status:")
print(
    orders.groupby("status")["delivered_date"]
    .apply(lambda column: column.isna().sum())
)

print("\nDelivered orders with missing delivery dates:")
print(
    ((orders["status"] == "Delivered") & orders["delivered_date"].isna()).sum()
)


Missing delivery dates by order status:
status
Cancelled     5996
Delivered        0
Processing    2501
Returned      3933
Shipped       5071
Name: delivered_date, dtype: int64

Delivered orders with missing delivery dates:
0


Order Status | Missing Delivery Dates | Interpretation

Cancelled | 5,996 | Missing delivery dates are expected because cancelled orders are generally not delivered.

Delivered | 0 | All delivered orders have a delivery date. No missing-date issue was detected for this status.

Processing | 2,501 | Missing delivery dates are expected because these orders are still being processed.

Returned | 3,933 | Missing delivery dates require further investigation to determine whether these records are valid.

Shipped | 5,071 | Missing delivery dates may be expected because shipped orders may not yet have been delivered.

Total | 17,501 | Matches our earlier result

Overall Conclusion: No delivered orders have missing delivery dates. The missing dates appear consistent with order status, although returned orders should be investigated further to confirm data consistency.

NOW LET'S CHECK WHETHER DELIVERY DATES OCCUR BEFORE ORDER DATES

In [94]:

invalid_delivery_dates = orders[
    orders["delivered_date"].notna()
    & (orders["delivered_date"] < orders["order_date"])
]

print("Orders delivered before order date:", len(invalid_delivery_dates))
print(invalid_delivery_dates[["order_id", "order_date", "delivered_date", "status"]].head(10))


Orders delivered before order date: 0
Empty DataFrame
Columns: [order_id, order_date, delivered_date, status]
Index: []


ALL ORDERS WITH A RECORDED DELIVERY DATE HAVE A DELIVERY DATE ON OR AFTER THEIR ORDER DATE. NOW LET'S CHECK UNUSUALLY HIGH ORDER AMOUNTS

In [95]:
high_value_orders = orders[orders["final_amount"] > 500000]

print("Orders with final_amount above ₹5 lakh:", len(high_value_orders))
print(high_value_orders[["order_id", "final_amount", "status"]].head(15))

Orders with final_amount above ₹5 lakh: 312
       order_id  final_amount      status
33    ORD000034     527178.05   Delivered
34    ORD000035     532061.13   Cancelled
306   ORD000307     550716.17   Cancelled
583   ORD000584     745667.68   Delivered
590   ORD000591     520293.72   Cancelled
763   ORD000764     540156.71   Delivered
985   ORD000986     512079.51     Shipped
1092  ORD001093     680009.23   Cancelled
1114  ORD001115     646223.50   Delivered
1178  ORD001179     513624.71  Processing
1424  ORD001425     601877.74   Delivered
1469  ORD001470     521435.45   Cancelled
1485  ORD001486     523420.62   Cancelled
1501  ORD001502     755160.93    Returned
1807  ORD001808     681058.74   Delivered


LET'S CHECK PAYMENT STATUS DISTRIBUTION

In [96]:
print(payments["status"].value_counts(dropna=False))

status
Success    48252
Failed      1748
Name: count, dtype: int64


AN ORDER CAN HAVE A PAYMENT STATUS THAT DIFFERS FROM ITS ORDER STATUS. LET'S INSPECT HOW MANY ORDERS HAVE FAILED PAYMENTS AND COMPARE THEIR ORDER STATUSES.

In [97]:
failed_payments = payments[payments["status"] == "Failed"]

print("Failed payment records:", len(failed_payments))
print("\nOrder statuses associated with failed payments:")
print(failed_payments["order_id"].map(orders.set_index("order_id")["status"]).value_counts(dropna=False))

Failed payment records: 1748

Order statuses associated with failed payments:


order_id
Cancelled    1748
Name: count, dtype: int64


ALL 1,748 FAILED PAYMENT RECORDS ARE ASSOCIATED WITH ORDERS WHOSE STATUS IS CANCELLED. THE NEXT CHECK VERIFIES REFERENTIAL INTEGRITY BETWEEN THE RETURNS AND ORDERS DATASETS. EACH RETURN SHOULD REFERENCE AN ORDER THAT EXISTS.

In [98]:
return_order_ids = set(returns["order_id"])
existing_order_ids = set(orders["order_id"])

unmatched_returns = returns[~returns["order_id"].isin(existing_order_ids)]

print("Total return records:", len(returns))
print("Returns linked to existing orders:", len(returns) - len(unmatched_returns))
print("Returns linked to missing orders:", len(unmatched_returns))

Total return records: 10000
Returns linked to existing orders: 10000
Returns linked to missing orders: 0


NOW WE'LL VERIFY WHETHER EVERY PRODUCT_ID IN THE INVENTORY DATASET EXISTS IN THE PRODUCTS DATASET.

In [99]:
existing_product_ids = set(products["product_id"])

unmatched_inventory = inventory[
    ~inventory["product_id"].isin(existing_product_ids)
]

print("Total inventory records:", len(inventory))
print("Inventory records linked to existing products:", len(inventory) - len(unmatched_inventory))
print("Inventory records linked to missing products:", len(unmatched_inventory))

Total inventory records: 1000
Inventory records linked to existing products: 1000
Inventory records linked to missing products: 0


NOW WE'LL CHECK INVENTORY STATUS DISTRIBUTION

In [100]:
print(inventory["status"].value_counts(dropna=False))

status
In Stock        922
Low Stock        76
Out of Stock      2
Name: count, dtype: int64


LET'S CHECK CUSTOMERS WITH ZERO ORDERS

In [101]:
zero_order_customers = customers[customers["total_orders"] == 0]

print("Customers with zero orders:", len(zero_order_customers))
print(zero_order_customers[["customer_id", "total_orders", "total_spent"]].head(15))

Customers with zero orders: 457
    customer_id  total_orders  total_spent
65    CUST00066             0          0.0
121   CUST00122             0          0.0
123   CUST00124             0          0.0
127   CUST00128             0          0.0
161   CUST00162             0          0.0
173   CUST00174             0          0.0
193   CUST00194             0          0.0
229   CUST00230             0          0.0
249   CUST00250             0          0.0
262   CUST00263             0          0.0
271   CUST00272             0          0.0
316   CUST00317             0          0.0
319   CUST00320             0          0.0
343   CUST00344             0          0.0
382   CUST00383             0          0.0


NEXT, WE'LL VERIFY THAT EVERY CUSTOMER_ID REFERENCED IN ORDERS EXISTS IN THE CUSTOMERS DATASET.

In [102]:
existing_customer_ids = set(customers["customer_id"])

unmatched_orders = orders[
    ~orders["customer_id"].isin(existing_customer_ids)
]

print("Total orders:", len(orders))
print("Orders linked to existing customers:", len(orders) - len(unmatched_orders))
print("Orders linked to missing customers:", len(unmatched_orders))

Total orders: 50000
Orders linked to existing customers: 50000
Orders linked to missing customers: 0


LET'S CHECK PRODUCT IDS IN ORDER ITEMS

In [103]:
existing_product_ids = set(products["product_id"])

unmatched_order_items = order_items[
    ~order_items["product_id"].isin(existing_product_ids)
]

print("Total order items:", len(order_items))
print("Items linked to existing products:", len(order_items) - len(unmatched_order_items))
print("Items linked to missing products:", len(unmatched_order_items))

Total order items: 100000
Items linked to existing products: 100000
Items linked to missing products: 0


NOW WE'LL  CHECK SUPPLIER IDS IN THE PRODUCTS DATASET

In [104]:
existing_supplier_ids = set(suppliers["supplier_id"])

unmatched_products = products[
    ~products["supplier_id"].isin(existing_supplier_ids)
]

print("Total products:", len(products))
print("Products linked to existing suppliers:", len(products) - len(unmatched_products))
print("Products linked to missing suppliers:", len(unmatched_products))

Total products: 1000
Products linked to existing suppliers: 1000
Products linked to missing suppliers: 0


NEXT, WE'LL IDENTIFY NEGATIVE VALUES IN IMPORTANT NUMERIC COLUMNS. THESE MAY BE LEGITIMATE IN SOME CONTEXTS, BUT NEGATIVE QUANTITIES OR PRICES COULD INDICATE DATA-QUALITY ISSUES.

In [105]:
numeric_columns = {
    "customers": ["age", "total_orders", "total_spent"],
    "inventory": ["quantity_available", "quantity_reserved", "unit_cost", "total_inventory_value"],
    "order_items": ["quantity", "unit_price", "gst_amount", "discount_amount", "total_price"],
    "orders": ["total_amount", "gst_amount", "shipping_charge", "discount_amount", "final_amount"],
    "payments": ["amount", "refund_amount"],
    "products": ["mrp", "selling_price", "cost_price", "weight_grams"],
    "returns": ["return_amount"],
    "suppliers": ["payment_terms_days"]
}

for name, columns in numeric_columns.items():
    for column in columns:
        negative_count = (datasets[name][column] < 0).sum()
        if negative_count > 0:
            print(f"{name}.{column}: {negative_count} negative values")

SINCE THE CELL PRODUCED NO OUTPUT, NONE OF THE NUMERIC COLUMNS WE CHECKED CONTAINED NEGATIVE VALUES. NEXT, WE'LL CHECK WHETHER IMPORTANT PRICE AND AMOUNT COLUMNS CONTAIN ZERO VALUES. ZERO VALUES AREN'T AUTOMATICALLY ERRORS, BUT THEY CAN REQUIRE INVESTIGATION.

In [106]:
price_columns = {
    "order_items": ["unit_price", "total_price"],
    "orders": ["total_amount", "final_amount"],
    "payments": ["amount"],
    "products": ["mrp", "selling_price", "cost_price"]
}

for name, columns in price_columns.items():
    for column in columns:
        zero_count = (datasets[name][column] == 0).sum()
        print(f"{name}.{column}: {zero_count} zero values")


order_items.unit_price: 0 zero values
order_items.total_price: 0 zero values
orders.total_amount: 0 zero values
orders.final_amount: 0 zero values
payments.amount: 0 zero values
products.mrp: 0 zero values
products.selling_price: 0 zero values
products.cost_price: 0 zero values


NEXT, WE'LL CALCULATE THE PAYMENT FAILURE RATE AS PART OF THE PHASE 1 QUALITY CHECKS.

In [107]:
total_payments = len(payments)
failed_payments_count = (payments["status"] == "Failed").sum()
failure_rate = (failed_payments_count / total_payments) * 100

print("Total payments:", total_payments)
print("Failed payments:", failed_payments_count)
print(f"Payment failure rate: {failure_rate:.2f}%")

Total payments: 50000
Failed payments: 1748
Payment failure rate: 3.50%


NOW, WE'LL COMPARE THE SUM OF ITEM-LEVEL TOTAL_PRICE FOR EACH ORDER WITH THE CORRESPONDING ORDERS.TOTAL_AMOUNT. THIS CAN HELP IDENTIFY INCONSISTENCIES BETWEEN THE TWO TABLES.

In [108]:
item_totals = order_items.groupby("order_id")["total_price"].sum()

order_comparison = orders.set_index("order_id").join(
    item_totals.rename("calculated_item_total")
)

order_comparison["amount_difference"] = (
    order_comparison["total_amount"] - order_comparison["calculated_item_total"]
)

print("Orders with item-total differences:", (order_comparison["amount_difference"].abs() > 0.01).sum())
print(order_comparison[order_comparison["amount_difference"].abs() > 0.01][
    ["total_amount", "calculated_item_total", "amount_difference"]
].head(10))

Orders with item-total differences: 0
Empty DataFrame
Columns: [total_amount, calculated_item_total, amount_difference]
Index: []


NEXT, WE'LL CHECK WHETHER FINAL_AMOUNT MATCHES THE CALCULATION:

TOTAL_AMOUNT + GST_AMOUNT + SHIPPING_CHARGE - DISCOUNT_AMOUNT

In [109]:
orders["calculated_final_amount"] = (
    orders["total_amount"]
    + orders["gst_amount"]
    + orders["shipping_charge"]
    - orders["discount_amount"]
)

orders["final_amount_difference"] = (
    orders["final_amount"] - orders["calculated_final_amount"]
)

mismatched_orders = orders[
    orders["final_amount_difference"].abs() > 0.01
]

print("Orders with final amount differences:", len(mismatched_orders))
print(mismatched_orders[
    ["order_id", "final_amount", "calculated_final_amount", "final_amount_difference"]
].head(10))

Orders with final amount differences: 50000
    order_id  final_amount  calculated_final_amount  final_amount_difference
0  ORD000001     100984.19                 96862.38                  4121.81
1  ORD000002     254478.01                276739.02                -22261.01
2  ORD000003      32557.06                 35423.12                 -2866.06
3  ORD000004      24836.94                 23899.88                   937.06
4  ORD000005     156689.29                173976.58                -17287.29
5  ORD000006      10370.03                 10609.06                  -239.03
6  ORD000007     306195.59                309422.18                 -3226.59
7  ORD000008     109938.66                111287.32                 -1348.66
8  ORD000009      72445.53                 79396.06                 -6950.53
9  ORD000010      56410.81                 52550.62                  3860.19


##### The result shows that all 50,000 orders have differences under our assumed formula. This doesn't necessarily mean the data is wrong — it may mean our assumption about how final_amount is calculated is incorrect.

##### For example, order ORD000001 has a difference of ₹4,121.81. Before flagging all orders as errors, let's inspect the project documentation and the relationship between the amount columns.

In [110]:
print(orders[[
    "total_amount",
    "gst_amount",
    "shipping_charge",
    "discount_amount",
    "final_amount"
]].head(10))

   total_amount  gst_amount  shipping_charge  discount_amount  final_amount
0     100984.19    16364.64                0         20486.45     100984.19
1     254478.01    41599.80                0         19338.79     254478.01
2      32557.06     3562.92                0           696.86      32557.06
3      24836.94     3092.88                0          4029.94      24836.94
4     156689.29    25092.36                0          7805.07     156689.29
5      10370.03     1502.82                0          1263.79      10370.03
6     306195.59    53769.96                0         50543.37     306195.59
7     109938.66    18234.45                0         16885.79     109938.66
8      72445.53    13081.11                0          6130.58      72445.53
9      56410.81     6901.58                0         10761.77      56410.81


THE OUTPUT REVEALS THE KEY ISSUE- FINAL_AMOUNT IS EXACTLY EQUAL TO TOTAL_AMOUNT IN THE FIRST 10 ROWS, EVEN THOUGH GST_AMOUNT AND DISCOUNT_AMOUNT ARE NON-ZERO.

SO OUR ASSUMED FORMULA DOESN'T MATCH THE WAY THESE COLUMNS ARE REPRESENTED IN YOUR DATASET. WE SHOULDN'T LABEL ALL 50,000 ORDERS AS ERRORS BASED ON THAT ASSUMPTION.

NOW, LET'S VERIFY THE RELATIONSHIP ACROSS ALL ORDERS

In [111]:
print("final_amount equals total_amount:",
      (orders["final_amount"] == orders["total_amount"]).sum())

print("Total orders:", len(orders))

final_amount equals total_amount: 49106
Total orders: 50000


LET'S EXAMINE THE 894 ORDERS WHERE THE AMOUNTS DIFFER


In [112]:
different_amount_orders = orders[
    orders["final_amount"] != orders["total_amount"]
]

print("Orders with different amounts:", len(different_amount_orders))
print(different_amount_orders[
    ["order_id", "total_amount", "gst_amount", "shipping_charge",
     "discount_amount", "final_amount"]
].head(10))

Orders with different amounts: 894
       order_id  total_amount  gst_amount  shipping_charge  discount_amount  \
173   ORD000174        241.98       38.88               40            12.90   
586   ORD000587        457.35        0.00               59            56.65   
665   ORD000666        389.84        0.00               59            53.16   
780   ORD000781        293.11        0.00               40            42.89   
841   ORD000842        405.11       20.85               59            32.74   
865   ORD000866        388.85        0.00               79            95.15   
1026  ORD001027        460.02        0.00               79            75.98   
1180  ORD001181        336.92        0.00               59            60.08   
1234  ORD001235        337.36       58.32               59            44.96   
1280  ORD001281        401.64        0.00               49            40.36   

      final_amount  
173         281.98  
586         516.35  
665         448.84  
780        

TESTING A FORMULA WITHOUT GST AND DISCOUNT:

In [113]:
orders["amount_without_gst_discount"] = (
    orders["total_amount"] + orders["shipping_charge"]
)

orders["amount_difference_check"] = (
    orders["final_amount"] - orders["amount_without_gst_discount"]
)

print(
    "Orders matching total_amount + shipping_charge:",
    (orders["amount_difference_check"].abs() <= 0.01).sum()
)

print(
    "Orders not matching this formula:",
    (orders["amount_difference_check"].abs() > 0.01).sum()
)

Orders matching total_amount + shipping_charge: 50000
Orders not matching this formula: 0


##### This confirms that all 50,000 orders satisfy the formula:

##### final_amount = total_amount + shipping_charge

GST and discounts are recorded separately and are not reflected in this formula for final_amount.

 So, we should use the correct final_amount formula in our working datase,i.e, final_amount
= total_amount + gst_amount + shipping_charge - discount_amount

and document the change. We'll preserve the original column values for traceability rather than lose the ability to compare them.

In [114]:
orders["final_amount_original"] = orders["final_amount"]

orders["final_amount"] = (
    orders["total_amount"]
    + orders["gst_amount"]
    + orders["shipping_charge"]
    - orders["discount_amount"]
)

print("Original final_amount values preserved:", "final_amount_original" in orders.columns)
print("Corrected final_amount sample:")
print(orders[[
    "order_id",
    "total_amount",
    "gst_amount",
    "shipping_charge",
    "discount_amount",
    "final_amount_original",
    "final_amount"
]].head(5))

Original final_amount values preserved: True
Corrected final_amount sample:
    order_id  total_amount  gst_amount  shipping_charge  discount_amount  \
0  ORD000001     100984.19    16364.64                0         20486.45   
1  ORD000002     254478.01    41599.80                0         19338.79   
2  ORD000003      32557.06     3562.92                0           696.86   
3  ORD000004      24836.94     3092.88                0          4029.94   
4  ORD000005     156689.29    25092.36                0          7805.07   

   final_amount_original  final_amount  
0              100984.19      96862.38  
1              254478.01     276739.02  
2               32557.06      35423.12  
3               24836.94      23899.88  
4              156689.29     173976.58  


LET'S VALIDATE THE CORRECTION

In [115]:
expected_final_amount = (
    orders["total_amount"]
    + orders["gst_amount"]
    + orders["shipping_charge"]
    - orders["discount_amount"]
)

difference = (orders["final_amount"] - expected_final_amount).abs()

print("Orders matching the formula:", (difference <= 0.01).sum())
print("Orders not matching the formula:", (difference > 0.01).sum())

Orders matching the formula: 50000
Orders not matching the formula: 0


ALL 50,000 ORDERS NOW MATCH THE DOCUMENTED FORMULA. THE CORRECTION IS VALIDATED. LET'S QUICKLY INSPECT THE SUMMARY STATISTICS OF THE CORRECTED FINAL_AMOUNT AND THE ORIGINAL VALUES TO UNDERSTAND THE IMPACT OF THE CORRECTION.

In [116]:
print(orders[[
    "final_amount_original",
    "final_amount"
]].describe())

       final_amount_original  final_amount
count           5.000000e+04  5.000000e+04
mean            6.294771e+04  6.548838e+04
std             9.789750e+04  1.054183e+05
min             1.360700e+02 -8.919904e+04
25%             6.893520e+03  6.470262e+03
50%             2.418338e+04  2.460369e+04
75%             7.149574e+04  7.312965e+04
max             1.133382e+06  1.217095e+06


LET'S IDENTIFY ORDERS WITH NEGATIVE CORRECTED AMOUNTS

In [117]:
negative_final_amounts = orders[orders["final_amount"] < 0]

print("Orders with negative final_amount:", len(negative_final_amounts))
print(negative_final_amounts[[
    "order_id",
    "total_amount",
    "gst_amount",
    "shipping_charge",
    "discount_amount",
    "final_amount"
]].head(10))

Orders with negative final_amount: 2052
        order_id  total_amount  gst_amount  shipping_charge  discount_amount  \
39930  ORD039931       6197.75     1020.96                0         13550.81   
39932  ORD039933       1559.38      240.30                0          3315.15   
39938  ORD039939       2726.19        0.00                0         12626.65   
39945  ORD039946        854.50        0.00                0           941.88   
39948  ORD039949        960.21        0.00                0          2597.19   
39950  ORD039951        276.26        0.00               49          5836.59   
39956  ORD039957        662.10        0.00                0          1661.57   
39961  ORD039962       1851.93      409.08                0          2733.96   
39964  ORD039965       3168.19        0.00                0          3776.79   
39967  ORD039968       1972.96      215.04                0          9878.36   

       final_amount  
39930      -6332.10  
39932      -1515.47  
39938      -9

2,052 ORDERS HAVE NEGATIVE CORRECTED FINAL_AMOUNT VALUES BECAUSE THEIR DISCOUNTS EXCEED THE SUM OF THE ORDER AMOUNT, GST, AND SHIPPING CHARGE. WE NEED TO DETERMINE WHETHER THE DISCOUNTS ARE INVALID OR WHETHER THE FORMULA OR SOURCE DATA NEEDS FURTHER INVESTIGATION.

CHECKING DISCOUNTS GREATER THAN THE SUBTOTAL:

In [118]:
excessive_discounts = orders[
    orders["discount_amount"] > orders["total_amount"]
]

print("Orders where discount exceeds total_amount:", len(excessive_discounts))
print(excessive_discounts[
    ["order_id", "total_amount", "gst_amount", "discount_amount", "final_amount"]
].head(10))

Orders where discount exceeds total_amount: 2197
        order_id  total_amount  gst_amount  discount_amount  final_amount
39930  ORD039931       6197.75     1020.96         13550.81      -6332.10
39932  ORD039933       1559.38      240.30          3315.15      -1515.47
39938  ORD039939       2726.19        0.00         12626.65      -9900.46
39945  ORD039946        854.50        0.00           941.88        -87.38
39948  ORD039949        960.21        0.00          2597.19      -1636.98
39950  ORD039951        276.26        0.00          5836.59      -5511.33
39956  ORD039957        662.10        0.00          1661.57       -999.47
39961  ORD039962       1851.93      409.08          2733.96       -472.95
39964  ORD039965       3168.19        0.00          3776.79       -608.60
39967  ORD039968       1972.96      215.04          9878.36      -7690.36


So as we can see, after the correction, we immediately discovered:
- 2,052 negative final_amount values.
- Discounts larger than the order value in many cases.
- The original dataset had no negative final amounts.
- The original data satisfied final_amount = total_amount + shipping_charge for all 50,000 orders.

These are strong indicators that our assumption may not actually match how this dataset was generated.
Therefore, I'll undo this correction for now. Otherwise, I'll end up introducing 2,052 new data-quality problems that weren't originally there.

In [119]:
orders["final_amount"] = orders["final_amount_original"]

LET'S VERIFY THE RESTORATION

In [120]:
print("Orders matching original final_amount:",
      (orders["final_amount"] == orders["final_amount_original"]).sum())

print("Total orders:", len(orders))

Orders matching original final_amount: 50000
Total orders: 50000


ALL 50,000 ORIGINAL FINAL_AMOUNT VALUES HAVE BEEN RESTORED SUCCESSFULLY.

NEXT, WE'LL VALIDATE CUSTOMER AGES. AGES THAT ARE ZERO OR NEGATIVE, OR UNUSUALLY HIGH.


In [121]:
invalid_ages = customers[
    (customers["age"] <= 0) | (customers["age"] > 100)
]

print("Customers with invalid ages:", len(invalid_ages))
print(invalid_ages[["customer_id", "age"]].head(10))

Customers with invalid ages: 0
Empty DataFrame
Columns: [customer_id, age]
Index: []


LET'S CHECK CUSTOMER AGE DISTRIBUTION

In [122]:
print(customers["age"].describe())


count    10000.00000
mean        41.67630
std         13.83438
min         18.00000
25%         30.00000
50%         42.00000
75%         54.00000
max         65.00000
Name: age, dtype: float64


NOW WE'LL CHECK WHETHER PRODUCT SELLING PRICES EXCEED MRP

In [123]:
price_issues = products[
    products["selling_price"] > products["mrp"]
]

print("Products with selling price above MRP:", len(price_issues))
print(price_issues[
    ["product_id", "product_name", "mrp", "selling_price"]
].head(10))

Products with selling price above MRP: 0
Empty DataFrame
Columns: [product_id, product_name, mrp, selling_price]
Index: []


NOW WE'LL CHECK THE NUMBER OF MISSING VALUES IN EACH DATASET. WE'VE INSPECTED SEVERAL SPECIFIC QUALITY ISSUES; THIS GIVES US A BROADER VIEW BEFORE FINISHING PHASE 1.

In [124]:
for name, df in datasets.items():
    missing_count = df.isnull().sum().sum()
    print(f"{name}: {missing_count} missing values")

customers: 0 missing values
inventory: 0 missing values
order_items: 0 missing values
orders: 17501 missing values
payments: 50000 missing values
products: 0 missing values
returns: 0 missing values
suppliers: 0 missing values


NOW LET'S CHECK WHETHER ANY ORDERS HAVE MISSING DELIVERY DATES DESPITE BEING MARKED AS DELIVERED

In [125]:
orders.loc[
    (orders["status"] == "Delivered") & (orders["delivered_date"].isna()),
    ["order_id", "status", "order_date", "delivered_date"]
]

,order_id,status,order_date,delivered_date


NEXT STEP: CHECK FOR RETURNED ORDERS WITH MISSING DELIVERY DATES

In [126]:
orders.loc[
    (orders["status"] == "Returned") & (orders["delivered_date"].isna()),
    ["order_id", "status", "order_date", "delivered_date"]
]

,order_id,status,order_date,delivered_date
14,ORD000015,Returned,2024-12-15,NaT
16,ORD000017,Returned,2023-10-25,NaT
37,ORD000038,Returned,2025-04-22,NaT
53,ORD000054,Returned,2025-06-13,NaT
71,ORD000072,Returned,2025-03-09,NaT
...,...,...,...,...
49933,ORD049934,Returned,2024-01-07,NaT
49941,ORD049942,Returned,2025-04-15,NaT
49944,ORD049945,Returned,2024-09-12,NaT
49964,ORD049965,Returned,2025-03-04,NaT


NEXT: CHECK WHETHER SHIPPED ORDERS HAVE MISSING DELIVERY DATES

In [127]:
orders.loc[
    (orders["status"] == "Shipped") & (orders["delivered_date"].isna()),
    ["order_id", "status", "order_date", "delivered_date"]
]

,order_id,status,order_date,delivered_date
6,ORD000007,Shipped,2025-04-12,NaT
7,ORD000008,Shipped,2024-08-12,NaT
12,ORD000013,Shipped,2024-12-16,NaT
22,ORD000023,Shipped,2024-09-15,NaT
24,ORD000025,Shipped,2024-05-23,NaT
...,...,...,...,...
49985,ORD049986,Shipped,2024-08-15,NaT
49989,ORD049990,Shipped,2025-03-13,NaT
49992,ORD049993,Shipped,2023-08-17,NaT
49993,ORD049994,Shipped,2024-01-17,NaT


NEXT STEP: CHECK PROCESSING ORDERS

In [128]:
orders.loc[
    (orders["status"] == "Processing") & (orders["delivered_date"].isna()),
    ["order_id", "status", "order_date", "delivered_date"]
]

,order_id,status,order_date,delivered_date
0,ORD000001,Processing,2025-05-22,NaT
15,ORD000016,Processing,2024-06-30,NaT
38,ORD000039,Processing,2024-03-15,NaT
44,ORD000045,Processing,2024-08-10,NaT
49,ORD000050,Processing,2023-10-09,NaT
...,...,...,...,...
49852,ORD049853,Processing,2025-02-03,NaT
49866,ORD049867,Processing,2025-02-14,NaT
49977,ORD049978,Processing,2024-01-09,NaT
49986,ORD049987,Processing,2023-11-23,NaT


NEXT: CHECK CANCELLED ORDERS

In [129]:
orders.loc[
    (orders["status"] == "Cancelled") & (orders["delivered_date"].isna()),
    ["order_id", "status", "order_date", "delivered_date"]
]

,order_id,status,order_date,delivered_date
2,ORD000003,Cancelled,2023-10-08,NaT
5,ORD000006,Cancelled,2025-03-02,NaT
10,ORD000011,Cancelled,2024-10-07,NaT
13,ORD000014,Cancelled,2024-08-20,NaT
34,ORD000035,Cancelled,2025-05-26,NaT
...,...,...,...,...
49947,ORD049948,Cancelled,2023-12-11,NaT
49956,ORD049957,Cancelled,2025-02-24,NaT
49981,ORD049982,Cancelled,2025-03-22,NaT
49983,ORD049984,Cancelled,2023-10-23,NaT


SUMMARIZING MISSING DELIVERY DATES BY ORDER STATUS

In [130]:
orders.groupby("status")["delivered_date"].apply(lambda x: x.isna().sum())

status
Cancelled     5996
Delivered        0
Processing    2501
Returned      3933
Shipped       5071
Name: delivered_date, dtype: int64

THE SUMMARY CONFIRMS THAT ALL 17,501 MISSING DELIVERY DATES ARE ACCOUNTED FOR ACROSS THE FIVE ORDER STATUSES.

ALTHOUGH THE MISSING DATES FOR RETURNED ORDERS (3,933) DESERVE FURTHER INVESTIGATION, SINCE WE WOULD GENERALLY EXPECT A RETURNED ORDER TO HAVE BEEN DELIVERED FIRST. WE'LL DOCUMENT THIS RATHER THAN FILLING IN DATES WITHOUT EVIDENCE.

NOW LET'S CHECK WHETHER ANY ORDERS HAVE DELIVERY DATES EARLIER THAN THEIR ORDER DATES

In [131]:
orders.loc[
    orders["delivered_date"] < orders["order_date"],
    ["order_id", "status", "order_date", "delivered_date"]
]

,order_id,status,order_date,delivered_date


WE ALREADY KNOW THAT ALL 50,000 VALUES IN PAYMENTS.REFUND_DATE ARE MISSING. BEFORE DECIDING HOW TO HANDLE THAT COLUMN, LET'S EXAMINE THE REFUND AMOUNTS.

In [132]:
payments[["status", "amount", "refund_amount", "refund_date"]].head(10)

,status,amount,refund_amount,refund_date
0,Success,100984.19,0,NaT
1,Success,254478.01,0,NaT
2,Success,32557.06,0,NaT
3,Success,24836.94,0,NaT
4,Success,156689.29,0,NaT
5,Success,10370.03,0,NaT
6,Success,306195.59,0,NaT
7,Success,109938.66,0,NaT
8,Success,72445.53,0,NaT
9,Success,56410.81,0,NaT


THIS SUGGESTS THE MISSING DATES MAY BE EXPECTED WHEN NO REFUND HAS OCCURRED, BUT WE SHOULD VERIFY THAT ACROSS THE FULL DATASET.

In [133]:
payments.groupby("status").agg(
    total_records=("payment_id", "count"),
    zero_refunds=("refund_amount", lambda x: (x == 0).sum()),
    positive_refunds=("refund_amount", lambda x: (x > 0).sum()),
    missing_refund_dates=("refund_date", lambda x: x.isna().sum())
)

,total_records,zero_refunds,positive_refunds,missing_refund_dates
status,,,,
Failed,1748,1748,0,1748
Success,48252,48252,0,48252


ALL 50,000 PAYMENT RECORDS HAVE REFUND_AMOUNT = 0 AND A MISSING REFUND_DATE, INCLUDING ALL 1,748 FAILED PAYMENTS.

NEXT: CHECK FAILED PAYMENT AMOUNTS

In [134]:
payments.loc[
    payments["status"] == "Failed",
    ["payment_id", "order_id", "status", "amount", "refund_amount", "refund_date"]
].head(10)

,payment_id,order_id,status,amount,refund_amount,refund_date
13,PAY000014,ORD000014,Failed,152164.56,0,NaT
41,PAY000042,ORD000042,Failed,23877.28,0,NaT
89,PAY000090,ORD000090,Failed,12897.65,0,NaT
128,PAY000129,ORD000129,Failed,5013.27,0,NaT
157,PAY000158,ORD000158,Failed,29647.72,0,NaT
164,PAY000165,ORD000165,Failed,27567.62,0,NaT
166,PAY000167,ORD000167,Failed,248924.74,0,NaT
222,PAY000223,ORD000223,Failed,111634.16,0,NaT
236,PAY000237,ORD000237,Failed,45149.50,0,NaT
350,PAY000351,ORD000351,Failed,50173.02,0,NaT


SINCE ALL FAILED PAYMENTS CORRESPOND TO CANCELLED ORDERS, WE SHOULD FLAG THIS FOR INVESTIGATION RATHER THAN ASSUME MONEY WAS ACTUALLY COLLECTED OR REFUNDED.

NOW WE'LL CHECK WHETHER ANY SUCCESSFUL PAYMENTS HAVE POSITIVE REFUND AMOUNTS

In [135]:
payments.loc[
    (payments["status"] == "Success") & (payments["refund_amount"] > 0),
    ["payment_id", "order_id", "amount", "refund_amount", "refund_date"]
]

,payment_id,order_id,amount,refund_amount,refund_date


THE OUTPUT IS AN EMPTY TABLE, CONFIRMING THAT NO SUCCESSFUL PAYMENTS HAVE A POSITIVE REFUND_AMOUNT.

WE'VE NOW ESTABLISHED THAT:
- ALL 50,000 PAYMENT RECORDS HAVE A REFUND AMOUNT OF ZERO.
- ALL 50,000 HAVE A MISSING REFUND_DATE.
- NO SUCCESSFUL PAYMENTS HAVE A POSITIVE REFUND AMOUNT.
- ALL 1,748 FAILED PAYMENTS CORRESPOND TO CANCELLED ORDERS.

WE'LL DOCUMENT THE MISSING REFUND DATES AS A DATA-QUALITY OBSERVATION AND PRESERVE THE ORIGINAL VALUES RATHER THAN FILLING THEM IN WITHOUT EVIDENCE.

NEXT: CHECK WHETHER ANY ORDERS HAVE MULTIPLE PAYMENT RECORDS

In [136]:
payments["order_id"].value_counts().value_counts().sort_index()

count
1    50000
Name: count, dtype: int64

NOW WE'LL CHECK WHETHER EVERY ORDER HAS A MATCHING PAYMENT RECORD

In [137]:
orders_without_payments = set(orders["order_id"]) - set(payments["order_id"])
payments_without_orders = set(payments["order_id"]) - set(orders["order_id"])

print("Orders without payments:", len(orders_without_payments))
print("Payments without matching orders:", len(payments_without_orders))

Orders without payments: 0
Payments without matching orders: 0


BOTH CHECKS RETURNED 0 UNMATCHED RECORDS, CONFIRMING THAT THE RELATIONSHIP BETWEEN ORDERS AND PAYMENTS IS VALID.

NEXT WE'LL CHECK WHETHER RETURN RECORDS CONTAIN DUPLICATE ORDER IDS

In [138]:
returns["order_id"].value_counts().value_counts().sort_index()

count
1    10000
Name: count, dtype: int64

LET'S CHECK WHETHER RETURN DATES OCCUR BEFORE THEIR CORRESPONDING ORDER DATES

In [139]:
return_date_check = returns.merge(
    orders[["order_id", "order_date"]],
    on="order_id",
    how="left"
)

return_date_check.loc[
    return_date_check["return_date"] < return_date_check["order_date"],
    ["order_id", "order_date", "return_date", "return_amount"]
]

,order_id,order_date,return_date,return_amount


NOW WE'LL CHECK WHETHER RETURN DATES ARE LATER THAN DELIVERY DATES

Since we previously found 3,933 Returned orders with missing delivery dates, let's check whether the remaining return records have return dates after their delivery dates.

In [140]:
return_date_check = return_date_check.merge(
    orders[["order_id", "delivered_date"]],
    on="order_id",
    how="left"
)

return_date_check.loc[
    return_date_check["return_date"] < return_date_check["delivered_date"],
    ["order_id", "delivered_date", "return_date"]
]

,order_id,delivered_date,return_date
5001,ORD028585,2024-04-17,2024-04-14
5006,ORD007893,2023-12-19,2023-12-16
5028,ORD041283,2025-02-22,2025-02-20
5033,ORD048411,2024-07-01,2024-06-27
5036,ORD032815,2023-10-09,2023-10-06
...,...,...,...
9972,ORD032758,2023-08-09,2023-08-06
9978,ORD035511,2024-04-12,2024-04-10
9984,ORD012700,2023-10-05,2023-10-03
9995,ORD028566,2025-04-25,2025-04-24


1,006 RETURN RECORDS HAVE A RETURN_DATE EARLIER THAN THEIR CORRESPONDING DELIVERED_DATE.

THIS IS A DATA-QUALITY ISSUE WORTH FLAGGING. A RETURN WOULD GENERALLY BE EXPECTED TO OCCUR ON OR AFTER DELIVERY, SO THESE RECORDS NEED INVESTIGATION. WE SHOULD PRESERVE THE ORIGINAL VALUES RATHER THAN CHANGING THEM WITHOUT EVIDENCE.

NEXT: CHECK HOW MANY RETURNED ORDERS HAVE MISSING DELIVERY DATES

WE PREVIOUSLY IDENTIFIED 3,933 RETURNED ORDERS WITH MISSING DELIVERY DATES. LET'S CONFIRM THAT COUNT DIRECTLY IN THE ORDERS DATASET.

In [141]:
(
    (orders["status"] == "Returned") &
    (orders["delivered_date"].isna())
).sum()

np.int64(3933)

So far, we've identified two return-related observations:

- 3,933 returned orders have missing delivered_date values.
- 1,006 return records have a return_date earlier than the corresponding delivery date.

NOW LET'S CHECK THE INVENTORY QUANTITY RELATIONSHIP

In [142]:
inventory.loc[
    inventory["quantity_reserved"] > inventory["quantity_available"],
    ["inventory_id", "product_id", "quantity_available", "quantity_reserved", "status"]
]

,inventory_id,product_id,quantity_available,quantity_reserved,status


THE EMPTY TABLE CONFIRMS THAT NO INVENTORY RECORDS HAVE QUANTITY_RESERVED GREATER THAN QUANTITY_AVAILABLE. THIS CHECK PASSES, SO NO CORRECTION IS NEEDED.

NEXT STEP: CHECK INVENTORY STATUS CONSISTENCY

LET'S VERIFY WHETHER THE INVENTORY STATUS COLUMN AGREES WITH THE AVAILABLE QUANTITY. FIRST, WE'LL INSPECT THE QUANTITIES ASSOCIATED WITH EACH STATUS.

In [143]:
inventory.groupby("status")["quantity_available"].agg(
    ["count", "min", "max"]
)

,count,min,max
status,,,
In Stock,922,15,500
Low Stock,76,1,46
Out of Stock,2,0,0


- IN STOCK: 922 RECORDS, QUANTITIES RANGE FROM 15 TO 500.
- LOW STOCK: 76 RECORDS, QUANTITIES RANGE FROM 1 TO 46.
- OUT OF STOCK: 2 RECORDS, BOTH HAVE QUANTITY 0.

HOWEVER, THE RANGES OVERLAP BETWEEN IN STOCK AND LOW STOCK, SO WE SHOULDN'T ASSUME A FIXED THRESHOLD WITHOUT CHECKING THE PROJECT'S DATA DICTIONARY.

NEXT STEP: CHECK FOR INCONSISTENT INVENTORY STATUSES

In [144]:
inventory.loc[
    ((inventory["status"] == "Out of Stock") & (inventory["quantity_available"] != 0)) |
    ((inventory["status"] != "Out of Stock") & (inventory["quantity_available"] == 0)),
    ["inventory_id", "product_id", "quantity_available", "status"]
]

,inventory_id,product_id,quantity_available,status


The empty table confirms that inventory stock status is consistent with available quantities under the rules we checked:

- No Out of Stock records have a nonzero quantity.
- No records with zero quantity have a different status.

No corrections are needed for this check.

Next step: Check for customers whose total_orders value doesn't match the actual number of orders

We previously found 457 customers with total_orders = 0. Now let's verify whether the recorded order counts match the orders dataset.

In [145]:
actual_orders = orders.groupby("customer_id")["order_id"].count()

customer_order_check = customers.set_index("customer_id").join(
    actual_orders.rename("actual_orders")
)

customer_order_check["actual_orders"] = customer_order_check["actual_orders"].fillna(0)

customer_order_check.loc[
    customer_order_check["total_orders"] != customer_order_check["actual_orders"],
    ["total_orders", "actual_orders"]
]

,total_orders,actual_orders
customer_id,,


Next step: Check whether customer spending matches order amounts

Let's compare each customer's recorded total_spent with the sum of their final_amount values across all orders.

In [146]:
actual_spending = orders.groupby("customer_id")["final_amount"].sum()

customer_spending_check = customers.set_index("customer_id").join(
    actual_spending.rename("actual_spending")
)

customer_spending_check["actual_spending"] = customer_spending_check["actual_spending"].fillna(0)

customer_spending_check["spending_difference"] = (
    customer_spending_check["total_spent"] - customer_spending_check["actual_spending"]
)

customer_spending_check.loc[
    customer_spending_check["spending_difference"].abs() > 0.01,
    ["total_spent", "actual_spending", "spending_difference"]
]

,total_spent,actual_spending,spending_difference
customer_id,,,


The empty table confirms that every customer's recorded total_spent matches the sum of their final_amount values across all orders, within the ₹0.01 tolerance.

Next step: Check whether product selling prices are below cost prices

We already checked that no product has a selling price above its MRP. Now let's identify products where the selling price is lower than the cost price, which may indicate a loss-making sale or a pricing decision worth investigating.

In [147]:
products.loc[
    products["selling_price"] < products["cost_price"],
    ["product_id", "product_name", "cost_price", "selling_price", "mrp"]
]

,product_id,product_name,cost_price,selling_price,mrp


The empty table confirms that no products have a selling_price below their cost_price. Based on these two columns, every product is priced at or above its cost.

Next: We'll check whether order-item totals are calculated correctly

We previously checked that the sum of order_items.total_price matches orders.total_amount for all 50,000 orders. Now let's verify the calculation within each order item.

In [148]:
order_items.loc[
    (order_items["quantity"] * order_items["unit_price"] - order_items["total_price"]).abs() > 0.01,
    ["item_id", "order_id", "quantity", "unit_price", "total_price"]
]

,item_id,order_id,quantity,unit_price,total_price
0,ITM0000001,ORD000001,1,40989,38046.23
1,ITM0000002,ORD000001,1,62532,61414.65
2,ITM0000003,ORD000001,1,1585,1523.31
3,ITM0000004,ORD000002,3,1107,3088.08
4,ITM0000005,ORD000002,2,114448,251389.93
...,...,...,...,...,...
99995,ITM0099996,ORD039928,2,4197,9454.97
99996,ITM0099997,ORD039929,2,1301,2399.17
99997,ITM0099998,ORD039929,2,6524,13449.92
99998,ITM0099999,ORD039929,1,52231,58127.99


The output shows 99,997 out of 100,000 order items differ from quantity × unit_price by more than ₹0.01.

This does not automatically mean those records are incorrect. The total_price column may account for discounts or taxes, so we need to inspect the calculation before drawing a conclusion.

Next step: Check the size of the differences

In [149]:
order_items["calculated_price"] = (
    order_items["quantity"] * order_items["unit_price"]
)

order_items["price_difference"] = (
    order_items["calculated_price"] - order_items["total_price"]
)

order_items["price_difference"].describe()

count    100000.000000
mean      -1540.838495
std        5897.450960
min      -80463.520000
25%        -985.515000
50%         -89.330000
75%         140.632500
max       11639.910000
Name: price_difference, dtype: float64

The summary shows that quantity × unit_price often differs from total_price, and the differences can be substantial. For example, the median difference is −₹89.33, meaning the calculated price is lower than the recorded total by ₹89.33 at the midpoint.

This suggests total_price may incorporate discounts, taxes, or another pricing rule. We shouldn't modify these values without confirming the intended formula.

After Checking the data dictionary, It defines order_items.total_price as the net price paid after GST minus discount. It also provides gst_amount and discount_amount separately.

So the documented calculation is:
total_price = quantity x unit_price + gst_amount - discount_amount

Next: let's validate the documented formula

In [150]:
order_items["expected_total"] = (
    order_items["quantity"] * order_items["unit_price"]
    + order_items["gst_amount"]
    - order_items["discount_amount"]
)

order_items["total_difference"] = (
    order_items["expected_total"] - order_items["total_price"]
)

order_items.loc[
    order_items["total_difference"].abs() > 0.01,
    ["item_id", "quantity", "unit_price", "gst_amount",
     "discount_amount", "total_price", "expected_total", "total_difference"]
]

,item_id,quantity,unit_price,gst_amount,discount_amount,total_price,expected_total,total_difference


The empty table confirms that all 100,000 order-item records match the documented pricing formula:

total_price = quantity x unit_price + gst_amount - discount_amount

Next step: Remove temporary validation columns

We created temporary columns during our checks. Let's remove them from order_items so they don't accidentally appear in the final cleaned dataset.

In [151]:
order_items.drop(
    columns=["calculated_price", "price_difference", "expected_total", "total_difference"],
    inplace=True,
    errors="ignore"
)

Let's move on to validating whether the quantity values in order_items are positive integers:

In [152]:
order_items.loc[
    (order_items["quantity"] <= 0) |
    (order_items["quantity"] % 1 != 0),
    ["item_id", "order_id", "product_id", "quantity"]
]

,item_id,order_id,product_id,quantity


LET'S NOW VERIFY THAT PRODUCT WEIGHTS ARE POSITIVE

In [153]:
products.loc[
    products["weight_grams"] <= 0,
    ["product_id", "product_name", "weight_grams"]
]

,product_id,product_name,weight_grams


NEXT STEP: CHECK WHETHER SUPPLIER PAYMENT TERMS ARE VALID

LET'S VERIFY THAT SUPPLIERS DON'T HAVE NEGATIVE PAYMENT TERMS.


In [154]:
suppliers.loc[
    suppliers["payment_terms_days"] < 0,
    ["supplier_id", "supplier_name", "payment_terms_days"]
]

,supplier_id,supplier_name,payment_terms_days


LET'S CHECK WHETHER ANY SUPPLIER HAS UNUSUALLY HIGH PAYMENT TERMS (MORE THAN 120 DAYS).

In [155]:
suppliers.loc[
    suppliers["payment_terms_days"] > 120,
    ["supplier_id", "supplier_name", "payment_terms_days"]
]

,supplier_id,supplier_name,payment_terms_days


NEXT: VALIDATE SUPPLIER RATINGS

In [156]:
suppliers.loc[
    (suppliers["rating"] < 1) | (suppliers["rating"] > 5),
    ["supplier_id", "supplier_name", "rating"]
]

,supplier_id,supplier_name,rating


FINAL DATA-QUALITY CHECK: VERIFY SUPPLIER CONTACT DETAILS

WE'LL CHECK FOR MISSING SUPPLIER EMAIL ADDRESSES OR PHONE NUMBERS.

In [157]:
suppliers.loc[
    suppliers["email"].isna() | suppliers["phone"].isna(),
    ["supplier_id", "supplier_name", "email", "phone"]
]

,supplier_id,supplier_name,email,phone


The empty table confirms that all suppliers have both email addresses and phone numbers populated. No missing contact details were found.

We've now completed the main supplier validation checks:
- Supplier ratings are within the 1–5 range.
- Payment terms do not exceed 120 days.
- Email addresses and phone numbers are present.

Next step: Save the cleaned datasets

Before saving, let's check whether any temporary validation columns remain in our DataFrames, so we don't accidentally include them in the cleaned CSV files.

In [158]:
for name, df in datasets.items():
    print(name, df.columns.tolist())

customers ['customer_id', 'first_name', 'last_name', 'email', 'phone', 'city', 'state', 'pincode', 'gender', 'age', 'segment', 'registration_date', 'last_login_date', 'is_verified', 'is_active', 'total_orders', 'total_spent']
inventory ['inventory_id', 'product_id', 'warehouse_location', 'quantity_available', 'quantity_reserved', 'reorder_level', 'reorder_quantity', 'last_restocked_date', 'unit_cost', 'total_inventory_value', 'status']
order_items ['item_id', 'order_id', 'product_id', 'product_name', 'category', 'quantity', 'unit_price', 'gst_rate', 'gst_amount', 'discount_amount', 'total_price']
orders ['order_id', 'customer_id', 'order_date', 'order_time', 'status', 'city', 'state', 'pincode', 'total_amount', 'gst_amount', 'shipping_charge', 'discount_amount', 'final_amount', 'payment_method', 'shipping_partner', 'tracking_id', 'delivered_date', 'is_cod', 'channel', 'calculated_final_amount', 'final_amount_difference', 'amount_without_gst_discount', 'amount_difference_check', 'final_

In [159]:
for name, df in datasets.items():
    print(f"\n{name.upper()}")
    print("-" * 40)
    for column in df.columns:
        print(column)


CUSTOMERS
----------------------------------------
customer_id
first_name
last_name
email
phone
city
state
pincode
gender
age
segment
registration_date
last_login_date
is_verified
is_active
total_orders
total_spent

INVENTORY
----------------------------------------
inventory_id
product_id
warehouse_location
quantity_available
quantity_reserved
reorder_level
reorder_quantity
last_restocked_date
unit_cost
total_inventory_value
status

ORDER_ITEMS
----------------------------------------
item_id
order_id
product_id
product_name
category
quantity
unit_price
gst_rate
gst_amount
discount_amount
total_price

ORDERS
----------------------------------------
order_id
customer_id
order_date
order_time
status
city
state
pincode
total_amount
gst_amount
shipping_charge
discount_amount
final_amount
payment_method
shipping_partner
tracking_id
delivered_date
is_cod
channel
calculated_final_amount
final_amount_difference
amount_without_gst_discount
amount_difference_check
final_amount_original

PAYMEN

the orders dataset contains temporary validation columns that we should remove before saving the cleaned CSV files.

I identified these extra columns in orders:
- calculated_final_amount
- final_amount_difference
- amount_without_gst_discount
- amount_difference_check
- final_amount_original

The first four are temporary validation columns. final_amount_original was created as a backup, but we have already restored the original final_amount values

In [160]:
orders.drop(
    columns=[
        "calculated_final_amount",
        "final_amount_difference",
        "amount_without_gst_discount",
        "amount_difference_check",
        "final_amount_original"
    ],
    inplace=True,
    errors="ignore"
)

In [161]:
print(orders.columns.tolist())

['order_id', 'customer_id', 'order_date', 'order_time', 'status', 'city', 'state', 'pincode', 'total_amount', 'gst_amount', 'shipping_charge', 'discount_amount', 'final_amount', 'payment_method', 'shipping_partner', 'tracking_id', 'delivered_date', 'is_cod', 'channel']


NOW LET'S SAVE ALL 8 CLEANED DATASETS

In [162]:
from pathlib import Path

cleaned_data_path = Path("../data/cleaned")
cleaned_data_path.mkdir(parents=True, exist_ok=True)

for name, df in datasets.items():
    df.to_csv(cleaned_data_path / f"{name}.csv", index=False)

print(f"Saved {len(datasets)} cleaned datasets to: {cleaned_data_path.resolve()}")

Saved 8 cleaned datasets to: C:\Users\DARSH\Internmo_IndiaKart_E-Commerce_Project\data\cleaned


LET'S VERIFY THE SAVED FILES

In [163]:
for name, df in datasets.items():
    file_path = cleaned_data_path / f"{name}.csv"
    saved_df = pd.read_csv(file_path)
    print(f"{name}: Memory rows = {len(df)}, Saved rows = {len(saved_df)}")

customers: Memory rows = 10000, Saved rows = 10000
inventory: Memory rows = 1000, Saved rows = 1000
order_items: Memory rows = 100000, Saved rows = 100000
orders: Memory rows = 50000, Saved rows = 50000
payments: Memory rows = 50000, Saved rows = 50000
products: Memory rows = 1000, Saved rows = 1000
returns: Memory rows = 10000, Saved rows = 10000
suppliers: Memory rows = 200, Saved rows = 200
